<a href="https://colab.research.google.com/github/gonzaloelejalde/piii-2025/blob/main/PrimerParcial/PrimerParcial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CANAL

Código Pc Administrador:

In [ ]:
# admin_monitor_estetico.py
import socket
import threading

HOST = '0.0.0.0'
CONTROL_PORT = 5050

esp_conn = None
esp_addr = None
esp_lock = threading.Lock()
modo_error = False  # Nueva variable global

def esp_acceptor():
    global esp_conn, esp_addr
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((HOST, CONTROL_PORT))
    server.listen(1)
    print(f"[ADMIN] Esperando conexión de la ESP en {HOST}:{CONTROL_PORT} ...")
    while True:
        conn, addr = server.accept()
        with esp_lock:
            if esp_conn:
                try:
                    esp_conn.close()
                except:
                    pass
            esp_conn = conn
            esp_addr = addr
        print(f"[ADMIN] ESP conectada desde {addr[0]}:{addr[1]}")
        threading.Thread(target=esp_receiver, args=(conn, addr), daemon=True).start()

def esp_receiver(conn, addr):
    try:
        while True:
            data = conn.recv(2048)
            if not data:
                break
            lines = data.split(b'\n')
            for line in lines:
                if not line:
                    continue
                if line.startswith(b'CANAL (crudo):'):
                    payload = line[len(b'CANAL (crudo):'):].strip()
                    hex_str = ' '.join(f'{b:02X}' for b in payload)
                    print("\n🛰️  --- MENSAJE DEL CANAL ---")
                    print(hex_str)
                    continue
                if b'[OK]' in line:
                    print("\n[INFO] Reenvío: ✅ Correcto")
                elif b'[ERROR]' in line:
                    print(f"\n[INFO] Reenvío: ❌ {line.decode(errors='ignore')}")
                elif b'MODO_ERROR_ON' in line:
                    global modo_error
                    modo_error = True
                    print("[ADMIN] ⚠️  Modo error ACTIVADO")
                elif b'MODO_ERROR_OFF' in line:
                    modo_error = False
                    print("[ADMIN] ✅ Modo error DESACTIVADO")
                else:
                    print(f"\n[INFO] {line.decode(errors='ignore')}")
    except Exception as e:
        print(f"[ERROR] Receiver: {e}")
    finally:
        with esp_lock:
            if esp_conn == conn:
                esp_conn = None
                esp_addr = None
        print(f"[ADMIN] Conexión cerrada desde {addr[0]}")

def enviar_a_esp(msg):
    with esp_lock:
        if not esp_conn:
            print("No hay ESP conectada.")
            return False
        try:
            esp_conn.sendall((msg + "\n").encode())
            return True
        except Exception as e:
            print("Error enviando a ESP:", e)
            return False

def main_menu():
    global modo_error
    while True:
        print("\n--- ADMINISTRADOR ---")
        with esp_lock:
            ip = esp_addr[0] if esp_addr else "NINGUNA"
        print("IP ESP:", ip)
        print("Modo error actual:", "ACTIVADO ✅" if modo_error else "DESACTIVADO ❌")
        print("1) Solicitar INFO de la ESP")
        print("2) Activar MODO ERROR (10% modificar mensaje)")
        print("3) Desactivar MODO ERROR")
        print("4) Salir")
        op = input("Opción: ").strip()
        if op == "1":
            enviar_a_esp("info")
        elif op == "2":
            modo_error = True
            enviar_a_esp("MODO_ERROR_ON")
            print("[ADMIN] ✅ Modo error ACTIVADO.")
        elif op == "3":
            modo_error = False
            enviar_a_esp("MODO_ERROR_OFF")
            print("[ADMIN] ❌ Modo error DESACTIVADO.")
        elif op == "4":
            break
        else:
            print("Opción inválida.")

if __name__ == "__main__":
    threading.Thread(target=esp_acceptor, daemon=True).start()
    main_menu()

Código Esp-32

In [1]:
# esp_intermedia_monitor.py - MicroPython para ESP32 (nodo intermedio)
import network
import socket
import time
import _thread
import random  # Agregado para modo error

# --- Config WiFi ---
SSID = "Fibertel WiFi957 2.4GHz"
PASSWORD = "0423460366"

# --- IPs y puertos ---
PC_ADMIN_IP = "192.168.0.16"    # IP de la PC administradora
CONTROL_PORT = 5050             # Puerto donde escucha la PC
CHANNEL_PORT = 5051             # Puerto donde escucha transmisores
RECEIVER_IP = "10.0.0.52"       # IP de la otra ESP
RECEIVER_PORT = 5052            # Puerto donde escucha la otra ESP

# --- Estados ---
pc_sock = None
pc_lock = _thread.allocate_lock()
modo_error = False  # Nuevo

# --- Conexión WiFi ---
wifi = network.WLAN(network.STA_IF)
wifi.active(True)
wifi.connect(SSID, PASSWORD)
print("Conectando a WiFi...")
while not wifi.isconnected():
    time.sleep(0.5)
print("✅ Conectado a WiFi. IP local:", wifi.ifconfig()[0])

# --- Función para introducir error ---
# Ahora: si hay dígitos 0..7 en el mensaje ASCII, cambia uno de esos dígitos por otro 0..7 distinto.
# Si no hay dígitos 0..7, intenta reemplazar un carácter imprimible. Si no, fallback por byte.
def introducir_error(data):
    if len(data) == 0:
        return data

    PROB = 0.3  # probabilidad de introducir error (30%)
    if random.random() > PROB:
        return data  # no se aplica en esta ocasión

    # Intento prioritario: cambiar dígitos 0..7 en mensajes ASCII
    try:
        # Verificamos si todos los bytes son ASCII (<128)
        if all(b < 128 for b in data):
            s = data.decode('ascii')
            # Buscar posiciones de dígitos 0..7
            digit_positions = [i for i, ch in enumerate(s) if ch in '01234567']
            if digit_positions:
                idx = random.choice(digit_positions)
                orig_ch = s[idx]
                # Elegir nuevo dígito 0..7 distinto del original
                choices = [c for c in '01234567' if c != orig_ch]
                new_ch = random.choice(choices)
                s_mod = s[:idx] + new_ch + s[idx+1:]
                print("⚠️ [ERROR] Dígito modificado posición", idx, ":", orig_ch, "->", new_ch)
                return s_mod.encode('ascii')

            # Si no hay dígitos 0..7, intentar cambiar un carácter imprimible (comportamiento secundario)
            printable_chars = (
                "0123456789"
                "abcdefghijklmnopqrstuvwxyz"
                "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
                " !\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~"
            )
            printable_positions = [i for i, ch in enumerate(s) if ch in printable_chars]
            if printable_positions:
                idx = random.choice(printable_positions)
                orig_ch = s[idx]
                choices = [c for c in printable_chars if c != orig_ch]
                new_ch = random.choice(choices)
                s_mod = s[:idx] + new_ch + s[idx+1:]
                print("⚠️ [ERROR] Carácter imprimible modificado posición", idx, ":", repr(orig_ch), "->", repr(new_ch))
                return s_mod.encode('ascii')

    except Exception as e:
        # Si falla el manejo ASCII, lo manejamos en el fallback por bytes
        print("[WARN] Error al intentar modificar ASCII:", e)

    # Fallback: modificar 1 byte al azar (comportamiento antiguo)
    try:
        idx = random.randrange(len(data))
        nuevo_byte = random.randint(0, 255)
        print("⚠️ [ERROR] Byte modificado (fallback) posición", idx, ":", data[idx], "->", nuevo_byte)
        data = data[:idx] + bytes([nuevo_byte]) + data[idx+1:]
    except Exception as e:
        print("[ERROR] No se pudo aplicar modificación por byte:", e)

    return data

# --- Comunicación con PC Administradora ---
def pc_control_client():
    global pc_sock, modo_error
    while True:
        try:
            s = socket.socket()
            s.connect((PC_ADMIN_IP, CONTROL_PORT))
            with pc_lock:
                pc_sock = s
            print("🖥️ Conectado con la PC administradora")
            s.sendall(("INFO:ESP_IP=" + wifi.ifconfig()[0] + "\n").encode())
            while True:
                data = s.recv(1024)
                if not data:
                    break
                cmd = data.decode().strip()
                if cmd == "MODO_ERROR_ON":
                    modo_error = True
                    print("[⚠️] Modo error ACTIVADO")
                elif cmd == "MODO_ERROR_OFF":
                    modo_error = False
                    print("[✅] Modo error DESACTIVADO")
        except Exception as e:
            print("[❌] Error conexión con Admin:", e)
        finally:
            try:
                s.close()
            except:
                pass
            with pc_lock:
                pc_sock = None
        time.sleep(5)

# --- Reenvío a otra ESP ---
# Ahora NO aplica introducir_error aquí: la modificación se aplica en canal_server()
def reenviar_a_esp(msg_bytes):
    try:
        c = socket.socket()
        c.connect((RECEIVER_IP, RECEIVER_PORT))
        c.sendall(msg_bytes)
        c.close()
        print("[➡️] Reenviado correctamente a la otra ESP")
        with pc_lock:
            if pc_sock:
                pc_sock.sendall(b"[OK] Reenviado correctamente\n")
    except Exception as e:
        print("[⚠️] No se pudo reenviar:", e)
        with pc_lock:
            if pc_sock:
                pc_sock.sendall(f"[ERROR] No se pudo reenviar: {e}\n".encode())

# --- Servidor canal (recibe transmisores) ---
def canal_server():
    s = socket.socket()
    s.bind(('', CHANNEL_PORT))
    s.listen(5)
    print(f"[📡] Esperando transmisores en puerto {CHANNEL_PORT}...")
    while True:
        try:
            conn, addr = s.accept()
            print("[TX] Conexión desde", addr)
            try:
                data = conn.recv(2048)
                if not data:
                    continue

                # 1) Mostrar mensaje crudo
                print("[TX] Mensaje recibido (crudo):", data)

                # 2) Generar versión modulada según modo_error (aquí se aplica el cambio)
                if modo_error:
                    msg_modulado = introducir_error(data)
                else:
                    msg_modulado = data

                # 3) Mostrar mensaje modulado
                print("[TX] Mensaje modulado:", msg_modulado)

                # 4) Reenviar mensaje modulado a la otra ESP
                reenviar_a_esp(msg_modulado)

                # 5) Enviar ambos a la PC (si está conectada)
                with pc_lock:
                    if pc_sock:
                        pc_sock.sendall(b"CANAL (crudo): " + data + b"\n")
                        pc_sock.sendall(b"CANAL (modulado): " + msg_modulado + b"\n")

            except Exception as e:
                print("[Error canal]", e)
            finally:
                conn.close()

        except Exception as e:
            print("[Error aceptando conexión]", e)
        time.sleep(0.05)

# --- Lanzar hilos ---
_thread.start_new_thread(pc_control_client, ())
_thread.start_new_thread(canal_server, ())

# --- Mantener vivo ---
while True:
    time.sleep(1)


ModuleNotFoundError: No module named 'network'